# 01 · Signal

*Part 1, section 2 and Part 2, section 1.* A trace says what happened. **Signal** says how it went.
It arrives from three places, and all of it lands on the trace as an **assessment** with a
source, so every later step reads and writes the same evidence:

| source | kind | here |
|---|---|---|
| HUMAN | feedback (`quality`: pass/fail + rationale), expectation (`expected_response`) | reviewers |
| CODE | deterministic checks | `examples/criteria.yaml` → `checks` |
| LLM_JUDGE | Guidelines judges, the `quality` judge | `criteria.yaml` → `guidelines`, `quality` |

In [1]:
import _nb as nb

experiment_id = nb.setup()

/Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


env file      /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/.env
tracking      sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/.selfimprove/notebooks/mlflow.db   experiment walkthrough
agent         python: examples.support_agent:answer
instructions  prompts:/walkthrough-instructions  (seed /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/harness/examples/instructions.md)
judge         anthropic:/claude-opus-5-5  (credentials found)
first pass    typesafe:/jev-latest  (re-judge p(pass) in 0.50-0.80)
improver      analyst   auto-promote False
mode          live


In [3]:
import mlflow
from selfimprove import feedback, signals, traces

week = traces.production()
print(len(week), "production traces")
t = week[0]
print("Q:", traces.question_of(t))
print("A:", traces.response_of(t)[:300])

29 production traces
Q: I lost my API key. Can you show it to me again?
A: Policies can vary by plan and region, so please double-check the Terms of Service too.

I can't show you your existing API key. I don't have access to your account's keys, and I couldn't find a policy for retrieving one. For security reasons, keys are usually shown only once, when they're created. I


**Under the hood:** `traces.production()` is one search and one filter: it drops every trace an
evaluation run created, so evaluation traffic never counts as production signal.

**MLflow in this step**

* `mlflow.search_traces(filter_string="tags.si_source = 'production'", order_by=["timestamp_ms DESC"],
  return_type="list")`: the production traces, newest first, as `Trace` objects
* trace metadata `mlflow.sourceRun`: MLflow sets it on traces that an evaluation run created; those are
  the ones dropped

## 1. Human feedback and expectations

Feedback says a trace is worth testing; an **expectation** says what correct means. A trace with
both is one call away from being a dataset record. Here is one label by hand, exactly as a review
UI or a thumbs-down button would log it:

In [4]:
from mlflow.entities import AssessmentSource, AssessmentSourceType

me = AssessmentSource(source_type=AssessmentSourceType.HUMAN, source_id="you")
refund = next(t for t in week if "refund" in traces.question_of(t).lower())

mlflow.log_feedback(
    trace_id=refund.info.trace_id, name="quality", value=False, source=me,
    rationale="Quotes the old 30-day window; it has been 14 days since July 2026.",
)
mlflow.log_expectation(
    trace_id=refund.info.trace_id, name="expected_response", source=me,
    value="Refunds are available within 14 days of purchase.",
)
print(traces.human_feedback(mlflow.get_trace(refund.info.trace_id)))

Feedback(name='quality', source=AssessmentSource(source_type='HUMAN', source_id='you'), trace_id='tr-895e76aab047b6f3f3b13cf9d27eaaa0', run_id=None, rationale='Quotes the old 30-day window; it has been 14 days since July 2026.', metadata=None, span_id=None, create_time_ms=1793217043584, last_update_time_ms=1793217043584, assessment_id='a-8a81ff1e8b6b4d8bb1f5fe2236661a37', error=None, expectation=None, feedback=FeedbackValue(value=False, error=None), issue=None, overrides=None, valid=True)


Nobody labels a whole week by hand in a notebook. `examples/reviewer.py` plays three reviewers
(maria, dev, sam) who share an answer key but grade with slightly different rubrics, on purpose:
that is what real reviewer pools do. In real use this is `selfimprove label` in a terminal, or
`selfimprove review-queue` for labeling in the MLflow UI.

In [5]:
from examples import reviewer

counts = reviewer.review_all(sample=0.75)   # label about 3/4 of the traffic
for name, (ok, bad) in counts.items():
    print(f"{name:<6} {ok:>3} pass {bad:>3} fail")

maria    0 pass   0 fail
dev      0 pass   0 fail
sam      0 pass   0 fail


**Under the hood:** each reviewer verdict is a call to the harness's `feedback.label()`, which does
what the cell before did by hand. The request, response and spans stay as they were; the evidence
accumulates next to them.

**MLflow in this step**

* `mlflow.log_feedback(name="quality", value=True/False, rationale=...,
  source=AssessmentSource(source_type="HUMAN", source_id=<reviewer>))`: the verdict, stored on the
  trace as an assessment
* `mlflow.log_expectation(name="expected_response", ...)`: for a failure, the right answer, with the same
  source

In [6]:
labeled = next(t for t in traces.production()
               if traces.human_feedback(t) is not None and t.info.trace_id != refund.info.trace_id)
for a in labeled.info.assessments:
    kind = "feedback" if a.feedback is not None else "expectation"
    print(f"{a.name:<18} {kind:<12} {str(a.value)[:45]:<46} {a.source.source_type}:{a.source.source_id}")

quality            feedback     False                                          HUMAN:maria
expected_response  expectation  No. Keys are shown only once. Create a new on  HUMAN:maria


## 2. Automated signal: checks and judges

From cheapest and most trustworthy to most flexible. The criteria file defines all of them; this is
what it builds:

In [7]:
print(open("examples/criteria.yaml").read()[:2200])

# What "good" means for the example agent. `selfimprove` builds its verifier from this file.
# Cheapest first: code checks, then LLM judges. See selfimprove/signals.py for every option.

# The alignable judge. Same name as the reviewers' feedback ("quality"), so the harness can
# measure how often it agrees with them and align it when it drifts (Part 3, section 5).
quality:
  instructions: |
    You are reviewing an answer from Acme Cloud's customer support agent.

    Question: {{ inputs }}
    Answer: {{ outputs }}

    Would a careful support lead send this answer to the customer as-is? Return true if it is
    correct, current, and gets to the point; otherwise false.
  # The same question for a first-pass decision model (SI_FAST_JUDGE_MODEL, e.g. Jev), phrased
  # as a yes/no. Verdicts in the uncertain band are re-judged with the instructions above.
  fast_instructions: >-
    Answer yes or no. Would a careful support lead send the answer in {{ outputs }} to the
    customer who as

In [8]:
for s in signals.live():
    print(f"{type(s).__name__:<22} {s.name}")
print("must-pass:", signals.must_pass())

CustomScorer           no_stale_policy
CustomScorer           no_disclaimer_opener
CustomScorer           looked_up_policy
CustomScorer           concise
CustomScorer           grounded_in_policy
Guidelines             leads_with_answer
Guidelines             no_guessing
CustomScorer           quality
must-pass: ('no_stale_policy',)


**Under the hood:** `signals.live()` only builds scorer objects; it calls no model and writes
nothing. The judges need credentials, so in offline mode the list holds only the code scorers.

**MLflow in this step**

* `@scorer`: each `checks` entry becomes a decorated function. `grounded_in_policy` is one too: it
  inspects TOOL spans and then asks a `make_judge()` judge
* `Guidelines(name=..., guidelines=...)`: one judge per `guidelines` entry
* `make_judge(name="quality", instructions=..., feedback_value_type=bool, model=SI_JUDGE_MODEL)`: the
  `quality` judge

A code check is a plain function over the response (or the whole trace, for tool checks):

In [9]:
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer

@scorer
def no_stale_refund_window(outputs) -> Feedback:
    stale = "30 day" in str(outputs).lower() or "30-day" in str(outputs).lower()
    return Feedback(value=not stale, rationale="Mentions the old 30-day window." if stale else "OK")

no_stale_refund_window(outputs="You can get a refund within 30 days of purchase.")

Feedback(name='feedback', source=AssessmentSource(source_type='CODE', source_id='default'), trace_id=None, run_id=None, rationale='Mentions the old 30-day window.', metadata=None, span_id=None, create_time_ms=1793217059510, last_update_time_ms=1793217059510, assessment_id=None, error=None, expectation=None, feedback=FeedbackValue(value=False, error=None), issue=None, overrides=None, valid=True)

### A cheap first pass: Jev (Part 3, section 3)

Reviewer labels arrive unevenly, and a text judge on every trace gets expensive. A **decision
model** such as TypeSafe's Jev (MLflow 3.17+) returns a yes/no with a **probability** instead of
generated text, cheap enough to judge most traffic. It gives no rationale, so verdicts in an
uncertain probability band go to the text judge, which supplies one. With
`SI_FAST_JUDGE_MODEL=typesafe:/jev-latest` and `TYPESAFE_API_KEY` in `.env`, the harness's
`quality` judge and the `grounded_in_policy` check both work this way.

In [10]:
from mlflow.genai.judges import make_judge
from selfimprove import config

if config.settings.has_fast_judge:
    jev = make_judge(
        name="quality",
        instructions="Answer yes or no. Would a careful support lead send the answer in {{ outputs }} "
                     "to the customer who asked {{ inputs }} as-is?",
        feedback_value_type=bool,
        model="typesafe:/jev-latest",
    )
    t = week[0]
    verdict = jev(inputs={"question": traces.question_of(t)}, outputs=traces.response_of(t))
    print("value:", verdict.value, "| p(pass):", signals.probability(verdict), "| rationale:", verdict.rationale)
    print("re-judged by the text judge when p(pass) is in", config.settings.fast_judge_band)
else:
    print("No first-pass model configured (SI_FAST_JUDGE_MODEL + TYPESAFE_API_KEY); the text judge does it all.")

value: False | p(pass): 0.15 | rationale: None
re-judged by the text judge when p(pass) is in (0.5, 0.8)


The grounded check goes one step further: it inspects the `lookup_policy` TOOL span first. A
missing, failed, or empty lookup fails the trace for free; only real lookups reach a model, which
is asked whether the answer stays within the policy text the tool returned.

In [11]:
import inspect
print(inspect.getsource(signals.make_grounded).split('"""', 2)[2][:1400])


    name, tool, needs = spec["name"], spec["tool"], spec.get("requires_input")
    instructions = spec["instructions"]
    text = make_judge(name=name, instructions=instructions, feedback_value_type=bool,
                      model=config.settings.judge_model) if config.settings.has_llm else None
    judge = cascade(fast_judge(name, spec.get("fast_instructions", instructions)), text) \
        if (text is not None and config.settings.has_fast_judge) else text

    input_match = spec.get("tool_input_match")

    @scorer(name=name)
    def grounded(trace) -> Feedback:
        from selfimprove import traces

        calls = traces.spans(trace, span_type=SpanType.TOOL, name=tool)
        if input_match:
            calls = [c for c in calls if re.search(input_match, json.dumps(c.inputs, default=str), re.I)]
        if not calls:
            return Feedback(value=False, rationale=f"The agent answered without calling {tool}.")
        ok = [c for c in calls if not failed_call(c)]
        i

## 3. Judge the week's traces in place

`evaluate()` with **no `predict_fn`** scores the existing traces: nothing is re-run, and the verdicts
land on the traces as assessments (source CODE or LLM_JUDGE) next to the reviewers'.
(`selfimprove score` does this for new traces since its last run.)

In [12]:
result = mlflow.genai.evaluate(data=week, scorers=signals.live())
week = traces.production()   # re-read: the traces now carry the verdicts

2026/10/28 12:51:13 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
Evaluating: 100%|██████████| 29/29 [Elapsed: 00:33, Remaining: 00:00] [predict_fn: 0%, scorers: 100%]



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: colorful-frog-977
  Run ID: 84593d0e4cb3475e9dc5c2c4b009e5c4

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



**Under the hood:** the harness's `score` job makes exactly this call on traces that have no verdict
yet.

**MLflow in this step**

* `mlflow.genai.evaluate(data=<traces>, scorers=...)` with no `predict_fn`: reads each trace's inputs,
  outputs and spans, runs every scorer, and logs each verdict back onto that same trace as a feedback
  assessment
* assessment source types: `@scorer` functions get `CODE`, judges get `LLM_JUDGE`
* the evaluation run: the call also creates an MLflow run that holds one aggregate metric per scorer
  (`<name>/mean`)

In [13]:
print("evaluation run:", result.run_id)
print({k: round(float(v), 2) for k, v in result.metrics.items()})
print()
for a in week[0].info.assessments:
    print(f"{a.name:<22} {str(a.value):<6} {a.source.source_type}:{a.source.source_id}")

evaluation run: 84593d0e4cb3475e9dc5c2c4b009e5c4
{'no_disclaimer_opener/mean': 0.66, 'looked_up_policy/mean': 0.86, 'concise/mean': 0.62, 'no_stale_policy/mean': 0.76, 'grounded_in_policy/mean': 0.0, 'quality/mean': 0.0, 'leads_with_answer/mean': 0.48, 'no_guessing/mean': 0.46}

quality                False  HUMAN:maria
expected_response      No. Keys are shown only once. Create a new one at Settings → Developer → API keys. HUMAN:maria
no_disclaimer_opener   False  CODE:default
looked_up_policy       True   CODE:default
concise                False  CODE:default
no_stale_policy        True   CODE:default
grounded_in_policy     False  LLM_JUDGE:typesafe:/jev-latest
quality                False  LLM_JUDGE:typesafe:/jev-latest
leads_with_answer      no     LLM_JUDGE:anthropic:/claude-opus-5-5
no_guessing            no     LLM_JUDGE:anthropic:/claude-opus-5-5


In [14]:
nb.verdicts(week, names=["quality", "no_stale_policy", "no_disclaimer_opener", "grounded_in_policy",
                          "leads_with_answer", "no_guessing"]).head(12)

,trace,v,question,quality,no_disclaimer_opener,no_stale_policy,grounded_in_policy,leads_with_answer,no_guessing
0,tr-48fa28a,1,I lost my API key. Can you show it to me agai,FAIL,FAIL,PASS,FAIL,FAIL,FAIL
1,tr-d574ce3,1,Does Acme Cloud integrate with Jira?,FAIL,FAIL,PASS,FAIL,FAIL,PASS
2,tr-2c5526f,1,How many teammates can I invite on Pro?,FAIL,PASS,PASS,FAIL,PASS,FAIL
3,tr-cd1624c,1,How do I export all my data?,FAIL,PASS,PASS,FAIL,PASS,PASS
4,tr-51a45d3,1,Can we set up SSO with Okta on our Pro plan?,FAIL,PASS,PASS,FAIL,PASS,PASS
5,tr-0f4e190,1,I lost my phone with my authenticator app. Ho,FAIL,FAIL,PASS,FAIL,FAIL,PASS
6,tr-b3dffc3,1,I typed my password wrong a bunch of times an,FAIL,PASS,PASS,FAIL,PASS,FAIL
7,tr-22dece3,1,I reset my password but still can't log in. W,FAIL,PASS,PASS,FAIL,FAIL,FAIL
8,tr-4654944,1,How long is a password reset link valid?,FAIL,PASS,FAIL,FAIL,PASS,FAIL
9,tr-a8696b1,1,What happens if my card payment fails?,FAIL,FAIL,PASS,FAIL,FAIL,FAIL


In [15]:
# With a first pass configured: how many quality verdicts did the text judge have to make?
rejudged = [a for t in week for a in t.info.assessments
            if a.name == "quality" and (a.metadata or {}).get("si.escalated") == "true"]
firsts = [a for t in week for a in t.info.assessments if a.name == "quality" and "si.escalated" in (a.metadata or {})]
if firsts:
    print(f"{len(rejudged)} of {len(firsts)} quality verdicts were re-judged by {config.settings.judge_model}")

3 of 29 quality verdicts were re-judged by anthropic:/claude-opus-5-5


In [16]:
must = signals.must_pass()
judged = [t for t in week if traces.has_verdict(t, must)]
print(f"failure rate on v1: {sum(traces.failed(t, must) for t in judged) / len(judged):.0%} of {len(judged)} traces")

failure rate on v1: 76% of 29 traces


## 4. Signal is searchable

Assessments are queryable like any other trace field. Boolean feedback is stored as text, so filter
with `'false'`, not `False`.

In [17]:
failing = mlflow.search_traces(
    filter_string="tags.si_source = 'production' AND feedback.quality = 'false' "
                  "AND expectation.expected_response IS NOT NULL",
    return_type="list",
)
print(len(failing), "traces failed by someone, with the right answer attached")

17 traces failed by someone, with the right answer attached


One caveat before trusting any of this: the `quality` judge writes under the **same name** as
the reviewers. That is deliberate (it lets the harness measure how often they agree, in
notebook 05), but it means a search for `feedback.quality = 'false'` matches judge verdicts too.
The harness's helpers (`traces.human_feedback`, `traces.failed`) always check the source.

**Next:** [02 · Store](02_store.ipynb), where the failures become a test suite.